# Model comparison on California-Housing-style data

This offline tutorial generates a deterministic **synthetic housing dataset** with the feature names and broad shapes used by California Housing examples. It does not download or reproduce the California Housing dataset.

A model-training partition fits two regressors. A disjoint discovery partition supplies per-row squared loss to Ginsu, and an untouched validation partition evaluates the candidate model's fixed discovered rules. We then compare two in-memory ``SliceAnalysis`` objects and examine stability across declared discovery subsamples.

All comparisons and recurrence summaries are descriptive. Overlapping slice totals are not additive attribution.

In [ ]:
import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import polars as pl
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

from ginsu import (
    DiscretizationPlan,
    QuantileBins,
    SliceAnalysis,
    Slicefinder,
    StabilityRun,
    compare_analyses,
    evaluate_stability,
)
from ginsu.plotting import plot_comparison, plot_search_report, plot_stability


def take_rows(frame: pl.DataFrame, indices: np.ndarray) -> pl.DataFrame:
    return frame.gather(sorted(int(index) for index in indices))

## Generate the offline regression fixture

The response combines income, location, age, and rooms with deliberately heterogeneous noise. Elevated loss for some occupancy and latitude ranges gives the slice workflow a known pattern to investigate.

In [ ]:
rng = np.random.default_rng(84)
row_count = 2_400
latitude = rng.uniform(32.5, 42.0, row_count)
longitude = rng.uniform(-124.3, -114.2, row_count)
median_income = np.clip(rng.lognormal(1.1, 0.45, row_count), 0.5, 15)
house_age = rng.integers(1, 53, row_count)
average_rooms = np.clip(rng.normal(5.5 + 0.25 * median_income, 1.4), 1, 15)
average_bedrooms = np.clip(rng.normal(1.1, 0.25, row_count), 0.3, 3)
population = rng.lognormal(7.0, 0.7, row_count)
average_occupancy = np.clip(rng.lognormal(1.0, 0.35, row_count), 1, 8)
coastal = (longitude < -121.5) & (latitude < 38.5)
noise_scale = (
    0.18 + 0.28 * (average_occupancy > 3.6) + 0.15 * (latitude > 39)
)
target = (
    0.45
    + 0.55 * median_income
    + 0.55 * coastal
    - 0.018 * house_age
    + 0.04 * average_rooms
    + rng.normal(0, noise_scale)
)
housing = pl.DataFrame(
    {
        "MedInc": median_income,
        "HouseAge": house_age.astype(float),
        "AveRooms": average_rooms,
        "AveBedrms": average_bedrooms,
        "Population": population,
        "AveOccup": average_occupancy,
        "Latitude": latitude,
        "Longitude": longitude,
    }
)
housing.head()

## Train two models outside the analysis partitions

Both models use exactly the same training rows. Discovery and validation squared losses are computed only after fitting.

In [ ]:
all_rows = np.arange(row_count)
model_rows, analysis_rows = train_test_split(
    all_rows, test_size=0.40, random_state=84
)
discovery_rows, validation_rows = train_test_split(
    analysis_rows, test_size=0.50, random_state=85
)
model_rows = np.sort(model_rows)
discovery_rows = np.sort(discovery_rows)
validation_rows = np.sort(validation_rows)
model_train = take_rows(housing, model_rows)
discovery = take_rows(housing, discovery_rows)
validation = take_rows(housing, validation_rows)

baseline_model = HistGradientBoostingRegressor(
    max_iter=60, max_leaf_nodes=15, random_state=84
).fit(model_train.to_numpy(), target[model_rows])
candidate_model = HistGradientBoostingRegressor(
    max_iter=140, max_leaf_nodes=31, random_state=84
).fit(model_train.to_numpy(), target[model_rows])
baseline_discovery_errors = (
    target[discovery_rows] - baseline_model.predict(discovery.to_numpy())
) ** 2
candidate_discovery_errors = (
    target[discovery_rows] - candidate_model.predict(discovery.to_numpy())
) ** 2
candidate_validation_errors = (
    target[validation_rows] - candidate_model.predict(validation.to_numpy())
) ** 2

pl.DataFrame(
    {
        "partition": ["model training", "slice discovery", "fixed-rule validation"],
        "rows": [len(model_rows), len(discovery_rows), len(validation_rows)],
    }
)

## Reuse one fitted discretization plan

Every continuous feature receives five error-independent quantile bins learned only from discovery features. The identical fitted plan supplies both model-error analyses and untouched validation.

In [ ]:
binning = DiscretizationPlan(
    numeric={name: QuantileBins(5) for name in housing.columns}
).fit(discovery)
discovery_binned = binning.transform(discovery)
validation_binned = binning.transform(validation)
finder_options = {
    "alpha": 0.95,
    "k": 8,
    "max_l": 2,
    "min_sup": 0.04,
    "verbose": False,
}
baseline_finder = Slicefinder(**finder_options).fit(
    discovery_binned, baseline_discovery_errors
)
candidate_finder = Slicefinder(**finder_options).fit(
    discovery_binned, candidate_discovery_errors
)
validation_report = candidate_finder.validate_slices(
    validation_binned, candidate_validation_errors, min_support=0.04
)
validation_report.statistics.select(
    "discovery_rank",
    "__ginsu_rule",
    "validation_status",
    "validation_support_fraction",
    "validation_error_lift",
).head(10)

## Compare declarative analysis artifacts

``SliceAnalysis.from_finder`` captures fitted result tables, search evidence, discretization, and caller-supplied provenance without raw observations or executable estimator state. We compare exact rules first and then deterministically pair related predicate sets. The example stays in memory; writing an artifact is always an explicit user action.

In [ ]:
baseline_analysis = SliceAnalysis.from_finder(
    baseline_finder,
    dataset_fingerprint="tutorial:synthetic-housing-discovery-v1",
    discretization=binning,
)
candidate_analysis = SliceAnalysis.from_finder(
    candidate_finder,
    dataset_fingerprint="tutorial:synthetic-housing-discovery-v1",
    discretization=binning,
)
comparison = compare_analyses(
    baseline_analysis,
    candidate_analysis,
    method="predicate",
    similarity_threshold=0.50,
)
comparison.summary

## Record recurrence across caller-controlled subsamples

Ginsu does not invent the resampling unit. Here each run is an explicitly seeded subsample of discovery rows, and its provenance remains attached. Three runs are useful for API illustration, not a strong stability claim.

In [ ]:
runs = []
for seed in (101, 202, 303):
    sample_rng = np.random.default_rng(seed)
    positions = np.sort(
        sample_rng.choice(discovery_binned.height, size=380, replace=False)
    )
    run_finder = Slicefinder(**finder_options).fit(
        take_rows(discovery_binned, positions),
        candidate_discovery_errors[positions],
    )
    runs.append(
        StabilityRun.from_finder(
            run_finder,
            run_id=f"subsample-{seed}",
            partition_id=f"synthetic-housing-{seed}",
            resampling_unit="synthetic_household",
            seed=seed,
        )
    )

stability = evaluate_stability(
    runs, minimum_successful_runs=3, stable_frequency=2 / 3
)
stability.summary.select(
    "__ginsu_rule", "selection_frequency_successful", "stability_status"
).head(10)

## Visual diagnostics

The comparison view retains emerged and resolved rules; it is not an additive waterfall. The stability view retains run evidence, and the search profile explains execution cost. In an interactive session, call ``figure.show()`` on a figure below.

In [ ]:
comparison_figure = plot_comparison(comparison, max_changes=30)
stability_figure = plot_stability(stability, max_slices=15)
search_figure = plot_search_report(candidate_finder.search_report_)

pl.DataFrame(
    {
        "figure": ["analysis comparison", "stability", "search profile"],
        "trace_count": [
            len(comparison_figure.data),
            len(stability_figure.data),
            len(search_figure.data),
        ],
    }
)

## Interpretation checklist

- Validate candidate rules on untouched losses before acting on discovery lift.
- Inspect emerged, resolved, and related rules separately in ``comparison.changes``.
- Do not sum metrics across overlapping slices as attribution.
- Treat unavailable stability runs as unknown evidence, never as rule absence.
- Replace the synthetic fixture with a governed dataset only after defining privacy, partition, reference-population, and provenance requirements.